# Emerging Technologies - Assessment Problems

This notebook explores the difference between classical and quantum algorithms through five problems centred on the [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm). The Deutsch-Jozsa algorithm is historically significant as one of the first demonstrations that a quantum computer can solve a well-defined problem exponentially faster than any classical deterministic algorithm.

The problems progress from classical Python to full Qiskit quantum circuits:

1. Generating random Boolean functions
2. Classical algorithm to determine function type
3. Quantum oracles for Deutsch's single-input problem
4. Deutsch's algorithm implemented in Qiskit
5. Generalisation to the full Deutsch-Jozsa algorithm for four-bit inputs

In [13]:
# Standard library: random selections and combinatorics.
import random
import itertools as it

# Quantum computing framework.
# See: https://docs.quantum.ibm.com/
import qiskit

# Quantum circuit simulator.
# See: https://qiskit.github.io/qiskit-aer/
import qiskit_aer as aer

# Quantum state and visualisation utilities.
import qiskit.quantum_info as info
import qiskit.visualization as viz

---

## Problem 1: Generating Random Boolean Functions

### Background

A **Boolean function** with $n$ inputs maps binary strings to a single bit:

$$f : \{0, 1\}^n \to \{0, 1\}$$

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm) is guaranteed that the function is one of two types:

- **Constant**: $f(x) = 0$ for all inputs, or $f(x) = 1$ for all inputs. There are exactly **2** constant functions.
- **Balanced**: $f(x) = 1$ for exactly **half** of all possible inputs. For $n = 4$ inputs ($2^4 = 16$ combinations), exactly 8 outputs must be 1. The number of balanced functions is $\binom{16}{8} = 12{,}870$.

As described in the [IBM Quantum course on classical information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/single-systems/classical-information#deterministic-operations), Boolean functions form the foundation of both classical and quantum computation.

### Implementation Strategy

The function uses a **lookup table** approach: all $2^4 = 16$ input combinations are precomputed, outputs are assigned (constant or balanced), and the result is returned as a Python **closure**. A closure is an inner function that retains access to variables from its enclosing scope - see [Real Python: Python Closures](https://realpython.com/python-closure/). This makes the function appear opaque to the caller: the type cannot be determined by inspecting the function object, only by evaluating it on inputs.

In [14]:
def random_constant_balanced():
    """Return a randomly chosen constant or balanced function.

    The returned function accepts exactly four Boolean arguments
    and returns 0 or 1.

    Constant: all 16 outputs are 0, or all are 1.
    Balanced: exactly 8 of the 16 outputs are 1.

    Returns:
        f (callable): A function f(a, b, c, d) -> int (0 or 1).
    """
    # Generate every 4-bit input combination: (0,0,0,0), (0,0,0,1), ..., (1,1,1,1).
    # itertools.product with repeat=4 gives 2^4 = 16 tuples.
    all_inputs = list(it.product((0, 1), repeat=4))

    # Randomly select function type with equal probability.
    func_type = random.choice(['constant', 'balanced'])

    if func_type == 'constant':
        # Constant: choose either always-0 or always-1.
        value = random.choice([0, 1])
        lookup = {inp: value for inp in all_inputs}
    else:
        # Balanced: exactly 8 outputs are 1, 8 are 0.
        # Start with an equal split, then shuffle to randomise placement.
        outputs = [0] * 8 + [1] * 8
        random.shuffle(outputs)
        lookup = dict(zip(all_inputs, outputs))

    def f(a, b, c, d):
        """Return f(a, b, c, d) from the precomputed lookup table."""
        return lookup[(a, b, c, d)]

    # Attach the type as metadata 
    f._type = func_type
    return f

### Demonstration

We generate a function and print its full truth table. The `f._type` attribute is attached for testing purposes only - in the real Deutsch-Jozsa scenario the caller has no way to inspect this.

In [15]:
# Generate a random constant-or-balanced function.
f_demo = random_constant_balanced()

# Print the full truth table (16 rows for n=4).
print(f"{'a':>3} {'b':>3} {'c':>3} {'d':>3}  |  f(a,b,c,d)")
print("-" * 32)
for inp in it.product((0, 1), repeat=4):
    print(f"{inp[0]:>3} {inp[1]:>3} {inp[2]:>3} {inp[3]:>3}  |  {f_demo(*inp)}")

  a   b   c   d  |  f(a,b,c,d)
--------------------------------
  0   0   0   0  |  0
  0   0   0   1  |  0
  0   0   1   0  |  0
  0   0   1   1  |  0
  0   1   0   0  |  0
  0   1   0   1  |  0
  0   1   1   0  |  0
  0   1   1   1  |  0
  1   0   0   0  |  0
  1   0   0   1  |  0
  1   0   1   0  |  0
  1   0   1   1  |  0
  1   1   0   0  |  0
  1   1   0   1  |  0
  1   1   1   0  |  0
  1   1   1   1  |  0


In [16]:
# Verify the output distribution.
all_outputs = [f_demo(*inp) for inp in it.product((0, 1), repeat=4)]
ones = sum(all_outputs)

print(f"Number of 1-outputs : {ones} / 16")
print(f"Number of 0-outputs : {16 - ones} / 16")
print(f"Hidden type         : {f_demo._type}")
print(f"Verified type       : {'constant' if ones in (0, 16) else 'balanced'}")

Number of 1-outputs : 0 / 16
Number of 0-outputs : 16 / 16
Hidden type         : constant
Verified type       : constant


In [17]:
# Generate several functions to confirm both types appear.
print("Generating 10 random functions:")
for i in range(10):
    g = random_constant_balanced()
    outputs = [g(*inp) for inp in it.product((0, 1), repeat=4)]
    ones_count = sum(outputs)
    verified = 'constant' if ones_count in (0, 16) else 'balanced'
    print(f"  Function {i+1}: ones={ones_count:>2}, type={verified}")

Generating 10 random functions:
  Function 1: ones=16, type=constant
  Function 2: ones= 8, type=balanced
  Function 3: ones=16, type=constant
  Function 4: ones= 8, type=balanced
  Function 5: ones=16, type=constant
  Function 6: ones= 8, type=balanced
  Function 7: ones=16, type=constant
  Function 8: ones=16, type=constant
  Function 9: ones=16, type=constant
  Function 10: ones= 8, type=balanced


---

## Problem 2: Classical Testing for Function Type

### Classical Query Complexity

**Query complexity** counts how many times an algorithm must call (query) a function $f$ to determine a property. This is the standard model used to compare classical and quantum algorithms - see [IBM Quantum: Deutsch-Jozsa, The Problem](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa#the-problem).

For a classical **deterministic** algorithm solving the constant-vs-balanced problem:

- After $2^{n-1}$ queries that all return the same value, the function could still be balanced (the remaining $2^{n-1}$ inputs could all return the opposite value).
- Only after the $(2^{n-1} + 1)$-th query returning the same value are we 100% certain the function is constant.
- Equivalently, if any two queries return different values, we are immediately certain the function is balanced.

For $n = 4$: **maximum queries needed = $2^3 + 1 = 9$**.

This contrasts with the quantum algorithm (Problem 4), which always uses exactly **1 query** - the central claim of [Deutsch & Jozsa, 1992](https://dl.acm.org/doi/10.5555/895843).

In [22]:
def determine_constant_balanced(f):
    """Determine whether f is constant or balanced.

    Queries f on all possible 4-bit inputs and returns the type.
    Uses early termination: as soon as two different outputs are
    seen, the function must be balanced (given the problem guarantee).

    Args:
        f (callable): A function f(a, b, c, d) -> int (0 or 1),
                      guaranteed to be constant or balanced.

    Returns:
        str: 'constant' or 'balanced'.
    """
    # All 16 possible inputs for n=4.
    all_inputs = list(it.product((0, 1), repeat=4))

    # Query f on the first input to establish a reference output.
    reference = f(*all_inputs[0])
    call_count = 1

    # Query remaining inputs; stop as soon as a differing output is found.
    for inp in all_inputs[1:]:
        output = f(*inp)
        call_count += 1

        # Different output found - must be balanced (problem guarantee).
        if output != reference:
            print(f"  Determined after {call_count} queries (early termination).")
            return 'balanced'

    # All 16 outputs were identical - function is constant.
    print(f"  Determined after {call_count} queries (all outputs identical).")
    return 'constant'

### Demonstration

In [19]:
# Test on 8 randomly generated functions and verify correctness.
print("Testing determine_constant_balanced on 8 random functions:\n")

for i in range(8):
    g = random_constant_balanced()
    expected = g._type  # Metadata attached in Problem 1 (cheating - for testing only).
    print(f"Function {i + 1} (hidden type: {expected}):")
    result = determine_constant_balanced(g)
    match = '✓' if result == expected else '✗'
    print(f"  Result: {result} {match}\n")

Testing determine_constant_balanced on 8 random functions:

Function 1 (hidden type: constant):
  Determined after 16 queries (all outputs identical).
  Result: constant ✓

Function 2 (hidden type: constant):
  Determined after 16 queries (all outputs identical).
  Result: constant ✓

Function 3 (hidden type: balanced):
  Determined after 4 queries (early termination).
  Result: balanced ✓

Function 4 (hidden type: constant):
  Determined after 16 queries (all outputs identical).
  Result: constant ✓

Function 5 (hidden type: balanced):
  Determined after 5 queries (early termination).
  Result: balanced ✓

Function 6 (hidden type: balanced):
  Determined after 5 queries (early termination).
  Result: balanced ✓

Function 7 (hidden type: constant):
  Determined after 16 queries (all outputs identical).
  Result: constant ✓

Function 8 (hidden type: balanced):
  Determined after 2 queries (early termination).
  Result: balanced ✓



### Worst-Case Analysis

In [21]:
# Demonstrate the worst case: a constant function requires all 16 queries
# before early termination is impossible (or 9 to be certain).

# Construct the worst-case manually: constant-0 function.
def const_zero_test(a, b, c, d):
    """Constant-0: always returns 0 - classical worst case."""
    return 0

print("Worst case - constant function (all outputs the same):")
result = determine_constant_balanced(const_zero_test)
print(f"Result: {result}")

Worst case - constant function (all outputs the same):
  Determined after 16 queries (all outputs identical).
Result: constant


### Efficiency Summary

| Scenario | Queries needed |
|---|---|
| Best case (balanced, different outputs on first two queries) | 2 |
| Worst case (constant, or balanced with all same outputs until last) | $2^{n-1} + 1 = 9$ |

**Maximum queries for 100% certainty: $2^{n-1} + 1 = 9$.**

The argument is as follows. After observing $2^{n-1}$ identical outputs, we cannot yet distinguish: the remaining $2^{n-1}$ inputs could all return the opposite value (balanced) or the same value (constant). One more query breaks this ambiguity. For a *probabilistic* classical algorithm (allowed to be wrong with some small probability $\varepsilon$), $O(1)$ queries also suffice - a subtlety noted in [Nielsen & Chuang, *Quantum Computation and Quantum Information*](https://www.cambridge.org/highereducation/books/quantum-computation-and-quantum-information/01E10196D0A682A6AEFFEA52D53BE9AE), Chapter 1. The **deterministic** separation is what makes the theorem significant.